
# Highway Networks — CIFAR-10 Implementation and Training

## 1. Building the Reusable Highway Layer

### 1.1 Objective

The goal is to implement and train a convolutional Highway Network for CIFAR-10 image classification.

We have already explored the Highway Network equations and verified that our Convolutional Highway Layer:

- Preserves input and output tensor dimensions.
- Uses independent transformation and gate parameters.
- Initially favors carrying information through negative gate bias initialization.
- Supports gradient-based optimization.

Now we move the reusable implementation into `src/highway.py`.

### 1.2 Highway Layer Equations

The Highway Layer computes:

$$
\boxed{y = H(x)T(x) + x(1-T(x))}
$$

For convolutional inputs:

$$
H(x)=\operatorname{ReLU}(\operatorname{Conv}_H(x))
$$

$$
T(x)=\sigma(\operatorname{Conv}_T(x))
$$

Both operations produce tensors with the same shape as the input.

### 1.3 Implementation Decisions

Our implementation uses:

| Component | Configuration |
|---|---|
| Transformation | `nn.Conv2d` + ReLU |
| Transform gate | `nn.Conv2d` + Sigmoid |
| Kernel size | $3\times3$ |
| Stride | 1 |
| Padding | 1 |
| Gate bias initialization | $-2.0$ |
| Carry gate | $1-T(x)$ |

The negative transform gate bias encourages the network to initially preserve more input information.

### 1.4 Separation of Responsibilities

- `src/highway.py` contains the reusable Highway Layer.
- `src/model.py` will contain the complete classification architecture.
- `03_cifar10_training.ipynb` will contain training experiments, evaluation, and visualizations.


In [1]:
import sys
from pathlib import Path

import torch
import torch.nn as nn

PROJECT_ROOT = Path.cwd().resolve().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.highway import ConvHighwayLayer

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)

Device: cuda


In [ ]:
torch.manual_seed(42)

layer = ConvHighwayLayer(
    channels=3,
    gate_bias=-2.0
).to(device)

x = torch.randn(4, 3, 32, 32, device=device)
y = layer(x)

print("Input shape:", x.shape)
print("Output shape:", y.shape)
print("Output device:", y.device)

assert x.shape == y.shape
assert torch.isfinite(y).all()

Input shape: torch.Size([4, 3, 32, 32])
Output shape: torch.Size([4, 3, 32, 32])
Output device: cuda:0
Test passed!
